In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/sample_submission.csv
/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/train.csv
/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/test.csv


In [2]:
import pandas as pd
import numpy as np

path = "/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/"

train = pd.read_csv(path + "train.csv")
test = pd.read_csv(path + "test.csv")
sample = pd.read_csv(path + "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample.shape)
train.head()

Train shape: (239, 14)
Test shape: (60, 13)
Sample submission shape: (60, 2)


,id,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,115,58.0,1,400,0,40,0,164000.0,1.0,139,0,0,91,0
1,23,53.0,0,63,1,60,0,368000.0,0.8,135,1,0,22,0
2,0,75.0,0,582,0,20,1,265000.0,1.9,130,1,0,4,1
3,247,64.0,0,143,0,25,0,246000.0,2.4,135,1,0,214,0
4,194,45.0,0,582,0,20,1,126000.0,1.6,135,1,0,180,1


In [3]:
target = "DEATH_EVENT"

X = train.drop(columns=["id", target])
y = train[target]
X_test = test.drop(columns=["id"])

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("Columns same?", list(X.columns) == list(X_test.columns))

X shape: (239, 12)
X_test shape: (60, 12)
Columns same? True


In [4]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

rf = RandomForestClassifier(n_estimators=300, random_state=42)
scores = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")

print("Average accuracy:", round(scores.mean(), 4))
print("Std (up-down):", round(scores.std(), 4))

Average accuracy: 0.8566
Std (up-down): 0.0539


In [5]:
rf.fit(X, y)
preds = rf.predict(X_test)

submission = pd.DataFrame({"id": test["id"], "DEATH_EVENT": preds})
submission.to_csv("submission.csv", index=False)

print(submission.shape)
print(submission["DEATH_EVENT"].value_counts())
submission.head()

(60, 2)
DEATH_EVENT
0    47
1    13
Name: count, dtype: int64


,id,DEATH_EVENT
0,274,0
1,149,0
2,120,0
3,226,0
4,125,0


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
}

for name, m in models.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="accuracy")
    print(f"{name}: {s.mean():.4f}  (+/- {s.std():.4f})")

LogReg: 0.8314  (+/- 0.0601)
RandomForest: 0.8566  (+/- 0.0539)
LightGBM: 0.8287  (+/- 0.0495)
CatBoost: 0.8482  (+/- 0.0509)


In [7]:
def add_features(df):
    df = df.copy()
    df["log_cpk"] = np.log1p(df["creatinine_phosphokinase"])
    df["log_creatinine"] = np.log1p(df["serum_creatinine"])
    df["creat_sodium"] = df["serum_creatinine"] / df["serum_sodium"]
    df["ef_age"] = df["ejection_fraction"] / df["age"]
    df["time_ef"] = df["time"] * df["ejection_fraction"]
    return df

X2 = add_features(X)
X_test2 = add_features(X_test)

for name, m in models.items():
    s = cross_val_score(m, X2, y, cv=cv, scoring="accuracy")
    print(f"{name} (new features): {s.mean():.4f}  (+/- {s.std():.4f})")

LogReg (new features): 0.8314  (+/- 0.0546)
RandomForest (new features): 0.8399  (+/- 0.0474)
LightGBM (new features): 0.8384  (+/- 0.0409)
CatBoost (new features): 0.8371  (+/- 0.0459)


In [8]:
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier(
    estimators=[(n, m) for n, m in models.items()],
    voting="soft"
)
s = cross_val_score(ensemble, X, y, cv=cv, scoring="accuracy")
print("Ensemble:", round(s.mean(), 4), "+/-", round(s.std(), 4))

Ensemble: 0.8412 +/- 0.0446


In [9]:
top = ["time", "serum_creatinine", "ejection_fraction", "age", "serum_sodium"]

for name, m in models.items():
    s = cross_val_score(m, X[top], y, cv=cv, scoring="accuracy")
    print(f"{name} (top 5 features): {s.mean():.4f}  (+/- {s.std():.4f})")

LogReg (top 5 features): 0.8341  (+/- 0.0495)
RandomForest (top 5 features): 0.8287  (+/- 0.0506)
LightGBM (top 5 features): 0.8218  (+/- 0.0558)
CatBoost (top 5 features): 0.8440  (+/- 0.0411)


In [10]:
for depth in [3, 5, 8, None]:
    for leaf in [1, 3, 5]:
        m = RandomForestClassifier(n_estimators=300, max_depth=depth,
                                   min_samples_leaf=leaf, random_state=42)
        s = cross_val_score(m, X, y, cv=cv, scoring="accuracy")
        print(f"depth={depth}, leaf={leaf}: {s.mean():.4f}")

depth=3, leaf=1: 0.8508
depth=3, leaf=3: 0.8522
depth=3, leaf=5: 0.8620
depth=5, leaf=1: 0.8496
depth=5, leaf=3: 0.8593
depth=5, leaf=5: 0.8593
depth=8, leaf=1: 0.8552
depth=8, leaf=3: 0.8580
depth=8, leaf=5: 0.8621
depth=None, leaf=1: 0.8566
depth=None, leaf=3: 0.8594
depth=None, leaf=5: 0.8621


In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=5, random_state=42)
proba = cross_val_predict(rf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1]

for t in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55]:
    print(t, round(accuracy_score(y, proba > t), 4))

0.3 0.8033
0.35 0.8159
0.4 0.8368
0.45 0.841
0.5 0.8661
0.55 0.8619


In [12]:
from sklearn.ensemble import VotingClassifier

ens = VotingClassifier([
    ("rf", RandomForestClassifier(n_estimators=500, min_samples_leaf=5, random_state=42)),
    ("cb", CatBoostClassifier(verbose=0, random_state=42)),
], voting="soft")
s = cross_val_score(ens, X, y, cv=cv, scoring="accuracy")
print("RF+CatBoost:", round(s.mean(), 4), "+/-", round(s.std(), 4))

RF+CatBoost: 0.858 +/- 0.0478


In [13]:
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier

others = {
    "ExtraTrees": ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3, random_state=42),
    "GB": GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, random_state=42),
    "CatBoost_small": CatBoostClassifier(verbose=0, depth=4, learning_rate=0.03, iterations=500, random_state=42),
}
for n, m in others.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="accuracy")
    print(n, round(s.mean(), 4))

ExtraTrees 0.8147
GB 0.8454
CatBoost_small 0.8441


In [14]:
final_rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=5, random_state=42)

s = cross_val_score(final_rf, X, y, cv=cv, scoring="accuracy")
print("Final RF CV:", round(s.mean(), 4), "+/-", round(s.std(), 4))

final_rf.fit(X, y)
preds = final_rf.predict(X_test)

submission = pd.DataFrame({"id": test["id"], "DEATH_EVENT": preds})
submission.to_csv("submission.csv", index=False)

print(submission.shape)
print(submission["DEATH_EVENT"].value_counts())

Final RF CV: 0.8635 +/- 0.0506
(60, 2)
DEATH_EVENT
0    46
1    14
Name: count, dtype: int64
